# Enterprise Energy Efficiency & Carbon Optimization Analysis

## Business Context
A multi-location enterprise is experiencing rising energy consumption and increasing sustainability pressure.

The objective of this analysis is to:
- Benchmark facility-level energy efficiency
- Identify peak usage patterns
- Estimate carbon footprint
- Simulate renewable energy impact
- Provide sustainability-focused recommendations

This notebook covers:
1. Data Understanding
2. Data Cleaning
3. Feature Engineering
4. Exploratory Analysis
5. KPI Modeling
6. Sustainability Insights

In [2]:

# Import Required Libraries


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load Dataset
df = pd.read_csv("building_energy_data_extended copy.csv")

# Preview first few rows
df.head()

,Timestamp,Building_ID,Energy_Usage (kWh),Temperature (°C),Humidity (%),Building_Type,Occupancy_Level
0,2025-01-01 00:00:00,B001,121.30,-7.20,79.36,Industrial,Low
1,2025-01-01 01:00:00,B001,230.76,12.62,80.37,Industrial,High
2,2025-01-01 02:00:00,B001,187.21,-1.33,37.74,Industrial,High
3,2025-01-01 03:00:00,B001,262.23,0.24,39.97,Industrial,High
4,2025-01-01 04:00:00,B001,472.97,5.44,89.29,Industrial,Medium


#### Data Understanding & Initial Profiling

Before cleaning the dataset, we examine its structure,
size, data types, and potential quality issues.

In [4]:
# Shape of dataset
df.shape

(7200, 7)

In [5]:
# Column names
df.columns

Index(['Timestamp', 'Building_ID', 'Energy_Usage (kWh)', 'Temperature (°C)',
       'Humidity (%)', 'Building_Type', 'Occupancy_Level'],
      dtype='object')

In [6]:
# Data types and null info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           7200 non-null   object 
 1   Building_ID         7200 non-null   object 
 2   Energy_Usage (kWh)  7200 non-null   float64
 3   Temperature (°C)    7200 non-null   float64
 4   Humidity (%)        7200 non-null   float64
 5   Building_Type       7200 non-null   object 
 6   Occupancy_Level     7200 non-null   object 
dtypes: float64(3), object(4)
memory usage: 393.9+ KB


###### Timestamp is object so requires cleaning

In [7]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [8]:
# Check basic statistics
df.describe()

,Energy_Usage (kWh),Temperature (°C),Humidity (%)
count,7200.000000,7200.000000,7200.000000
mean,277.869099,12.450781,59.826208
std,129.693860,13.033681,17.275891
min,50.050000,-9.990000,30.000000
25%,165.965000,1.017500,44.957500
50%,279.410000,12.550000,59.670000
75%,387.527500,23.772500,74.860000
max,499.920000,35.000000,89.990000


In [9]:
# Convert Timestamp to datetime format
df['Timestamp'] = pd.to_datetime(df['Timestamp'])


df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Timestamp           7200 non-null   datetime64[ns]
 1   Building_ID         7200 non-null   object        
 2   Energy_Usage (kWh)  7200 non-null   float64       
 3   Temperature (°C)    7200 non-null   float64       
 4   Humidity (%)        7200 non-null   float64       
 5   Building_Type       7200 non-null   object        
 6   Occupancy_Level     7200 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 393.9+ KB


### Time Feature Engineering

To analyze seasonal patterns, peak hours, and behavioral trends,
we extract additional time-based features from the timestamp column.

In [10]:
# Extract time features
df['Year'] = df['Timestamp'].dt.year
df['Month'] = df['Timestamp'].dt.month
df['Day'] = df['Timestamp'].dt.day
df['Hour'] = df['Timestamp'].dt.hour
df['Weekday'] = df['Timestamp'].dt.weekday

# Create Weekend flag
df['Is_Weekend'] = df['Weekday'].apply(lambda x: 1 if x >= 5 else 0)

df.head()

,Timestamp,Building_ID,Energy_Usage (kWh),Temperature (°C),Humidity (%),Building_Type,Occupancy_Level,Year,Month,Day,Hour,Weekday,Is_Weekend
0,2025-01-01 00:00:00,B001,121.30,-7.20,79.36,Industrial,Low,2025,1,1,0,2,0
1,2025-01-01 01:00:00,B001,230.76,12.62,80.37,Industrial,High,2025,1,1,1,2,0
2,2025-01-01 02:00:00,B001,187.21,-1.33,37.74,Industrial,High,2025,1,1,2,2,0
3,2025-01-01 03:00:00,B001,262.23,0.24,39.97,Industrial,High,2025,1,1,3,2,0
4,2025-01-01 04:00:00,B001,472.97,5.44,89.29,Industrial,Medium,2025,1,1,4,2,0


In [11]:
df['Hour'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23], dtype=int32)

#### Time Granularity Insight

The dataset contains hourly energy readings (0–23 hours).

This allows:
- Peak hour detection
- Daily load pattern analysis
- Seasonal comparison
- Renewable energy offset simulation during peak hours

In [12]:
# Undertsand Building Structure
df['Building_ID'].nunique()

20

#### Building Structure Insight

The dataset contains 20 unique buildings, representing a multi-location enterprise.

This enables:
- Cross-building energy efficiency benchmarking
- Carbon emission comparison
- Identification of high-consumption facilities
- Facility-level renewable integration simulation

In [13]:
# Building Type Distribution
df['Building_Type'].value_counts()

Building_Type
Industrial     2160
Commercial     1800
Educational    1800
Residential    1440
Name: count, dtype: int64

Industrial buildings represent the largest share of records, suggesting potentially higher operational energy intensity.


In [14]:
# Occupancy Structure
df['Occupancy_Level'].value_counts()

Occupancy_Level
Low       2432
High      2414
Medium    2354
Name: count, dtype: int64

The distribution is nearly balanced, allowing reliable comparison of energy usage patterns across different occupancy intensities.

##### KPI's

In [15]:
# Total Enterprise Energy Consumption

total_energy = df['Energy_Usage (kWh)'].sum()

print(f"Total Energy Consumption: {total_energy:,.2f} kWh")

Total Energy Consumption: 2,000,657.51 kWh


The total energy consumption across all buildings is approximately **2.0 million kWh** over the observed period.
This serves as the baseline for:
- Carbon footprint estimation
- Efficiency benchmarking
- Renewable energy offset simulation

In [16]:
# Energy Consumption by Building

energy_per_building = df.groupby('Building_ID')['Energy_Usage (kWh)'].sum().sort_values(ascending=False)
energy_per_building

Building_ID
B014    103663.54
B008    103135.37
B002    102672.52
B009    102214.57
B015    102083.78
B020    101719.75
B010    101476.33
B013    101395.11
B019    101077.83
B017    101023.50
B012    100411.05
B011     99359.19
B007     99043.12
B005     98938.54
B003     98027.94
B016     97550.02
B018     97191.75
B006     97068.87
B004     97043.78
B001     95560.95
Name: Energy_Usage (kWh), dtype: float64



Building **B014** has the highest total energy consumption,
followed closely by B008 and B002.

Although the variation across buildings is moderate,
these high-consumption facilities will be prioritized
for efficiency benchmarking and carbon impact analysis.

In [17]:
# Energy Consumption by Building Type

df.groupby('Building_Type')['Energy_Usage (kWh)'].sum().sort_values(ascending=False)

Building_Type
Industrial     589082.11
Commercial     502247.00
Educational    498137.22
Residential    411191.18
Name: Energy_Usage (kWh), dtype: float64

Industrial buildings consume the highest total energy.
However, total consumption alone does not indicate inefficiency,
as operational intensity varies by building type.
Therefore, energy intensity benchmarking will be performed next.

In [18]:
# Energy Intensity Benchmark (Average Usage per Hour)

df.groupby('Building_Type')['Energy_Usage (kWh)'].mean().sort_values(ascending=False)

Building_Type
Residential    285.549431
Commercial     279.026111
Educational    276.742900
Industrial     272.723199
Name: Energy_Usage (kWh), dtype: float64



While Industrial buildings have the highest total energy consumption,
Residential buildings show the highest average hourly energy usage.

This suggests that energy intensity patterns vary across building types
and total consumption alone is not a sufficient efficiency indicator.

#### Carbon Emission Estimation

To estimate carbon footprint, we apply an electricity grid emission factor.

Assumption:
We use a global average grid emission factor of **0.475 kg CO₂ per kWh**
based on widely cited international energy statistics.

This enables estimation of carbon impact associated with electricity consumption.

In [19]:
#Calculate Carbon Emissions

# Define emission factor
emission_factor = 0.475  # kg CO2 per kWh

# Calculate carbon emissions
df['Carbon_Emission_kg'] = df['Energy_Usage (kWh)'] * emission_factor

# Total carbon emissions
total_carbon = df['Carbon_Emission_kg'].sum()

print(f"Total Estimated Carbon Emissions: {total_carbon:,.2f} kg CO2")

Total Estimated Carbon Emissions: 950,312.32 kg CO2


In [20]:
# Total Carbon Footprint

print(f"Total Estimated Carbon Emissions: {total_carbon/1000:,.2f} metric tons CO2")

Total Estimated Carbon Emissions: 950.31 metric tons CO2


This quantifies the environmental impact of enterprise-wide electricity consumption
and establishes the baseline for decarbonization strategy.

In [21]:
# Carbon Hotspot Identification

#Carbon Emission by Building

carbon_by_building = df.groupby('Building_ID')['Carbon_Emission_kg'].sum().sort_values(ascending=False)

carbon_by_building

Building_ID
B014    49240.18150
B008    48989.30075
B002    48769.44700
B009    48551.92075
B015    48489.79550
B020    48316.88125
B010    48201.25675
B013    48162.67725
B019    48011.96925
B017    47986.16250
B012    47695.24875
B011    47195.61525
B007    47045.48200
B005    46995.80650
B003    46563.27150
B016    46336.25950
B018    46166.08125
B006    46107.71325
B004    46095.79550
B001    45391.45125
Name: Carbon_Emission_kg, dtype: float64



Building **B014** generates the highest carbon emissions,
followed closely by B008 and B002.

These facilities represent priority targets for:
- Energy efficiency interventions
- Renewable energy integration
- Carbon reduction initiatives

In [22]:
# Renewable Energy Simulation (20% Adoption Scenario)

renewable_percentage = 0.20

df['Adjusted_Energy_kWh'] = df['Energy_Usage (kWh)'] * (1 - renewable_percentage)
df['Adjusted_Carbon_kg'] = df['Adjusted_Energy_kWh'] * emission_factor

# Calculate new total emissions
new_total_carbon = df['Adjusted_Carbon_kg'].sum()

carbon_reduction = total_carbon - new_total_carbon

print(f"New Total Carbon Emissions: {new_total_carbon/1000:,.2f} metric tons CO2")
print(f"Estimated Carbon Reduction: {carbon_reduction/1000:,.2f} metric tons CO2")

New Total Carbon Emissions: 760.25 metric tons CO2
Estimated Carbon Reduction: 190.06 metric tons CO2




If 20% of electricity consumption is replaced with renewable energy:

- New estimated carbon emissions: **~760 metric tons CO₂**
- Estimated reduction: **~190 metric tons CO₂**

This demonstrates that partial renewable integration can significantly reduce enterprise-wide carbon footprint.



After completing data cleaning and feature engineering in Python,
the enriched dataset is exported to a SQL environment.

SQL is used for structured KPI aggregation,
ranking analysis and dashboard-ready reporting.

In [27]:
# Clean column names
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '_', regex=True)

df.columns

Index(['Timestamp', 'Building_ID', 'Energy_Usage_kWh_', 'Temperature_C_',
       'Humidity_', 'Building_Type', 'Occupancy_Level', 'Year', 'Month', 'Day',
       'Hour', 'Weekday', 'Is_Weekend', 'Carbon_Emission_kg',
       'Adjusted_Energy_kWh', 'Adjusted_Carbon_kg'],
      dtype='object')

In [28]:
df.to_csv("clean_energy_data.csv", index=False, encoding="utf-8")